# Bayesian Neural Network (BNN) — reproduction of Semenova et al. (2020)

Replicates the **proposed model** from:

> Semenova, Williams, Afzal & Lazic (2020), *A Bayesian neural network for toxicity
> prediction*, Computational Toxicology 16:100133 (`original_research.pdf`).

The paper's BNN is compared against the POLR baseline (see `baseline.ipynb`).
Model (paper Eq. 2), with a single hidden layer of **15 ReLU nodes** and an
ordered-logistic output:

```
h  = ReLU(X w01)
eta = h w12
w  = [w01, w12] ~ Normal(0, sigma^2)
sigma ~ HalfNormal(1)
y ~ OrderedLogistic(eta, c),   c ~ Normal(0, 20)
```

Notes:
- `X` is the **8 standardised main effects only** (147 x 8) — the BNN learns the
  interactions instead of using the POLR's 21 hand-built products.
- Eq. 2 has **no bias terms**; that is followed here.
- The paper used Julia/Turing + Flux.jl; this notebook uses PyMC. Same model, same
  NUTS sampler, different implementation.

## Colab

This notebook is self-contained. In Colab:
1. Run the *Install dependencies* cell.
2. Run the *Load data* cell and upload `train_df.parquet` and `test_df.parquet`
   (from `data/01_raw/`), or mount Google Drive.
3. Run all cells. The full model takes well under a minute on CPU; the optional
   20-bootstrap loop takes a few minutes.


In [ ]:
# Install dependencies (only needed in Colab / a fresh environment)
import importlib
import subprocess
import sys

for pkg, mod in [("pymc", "pymc"), ("arviz", "arviz"),
                 ("scikit-learn", "sklearn"), ("pytensor", "pytensor")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("dependencies ready")


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
from scipy.special import expit
from sklearn.preprocessing import StandardScaler

MAIN_FEATURES = ["ClogP", "BSEP", "Glu", "Glu_Gal", "THLE", "HepG2", "Fsp3", "log10cmax"]
HIDDEN = 15

CONFIG = dict(draws=2000, tune=1000, chains=4, target_accept=0.9, seed=42)


## 1. Load data

147 train / 37 test compounds; severity 1 (safe), 2 (moderate), 3 (most DILI concern).

In [ ]:
def load_data():
    for base in ["../data/01_raw", "data/01_raw", "."]:
        p = os.path.join(base, "train_df.parquet")
        if os.path.exists(p):
            return (pd.read_parquet(p),
                    pd.read_parquet(os.path.join(base, "test_df.parquet")))
    try:
        from google.colab import files
        print("Upload train_df.parquet and test_df.parquet")
        up = files.upload()
        names = list(up)
        tr = pd.read_parquet([n for n in names if "train" in n][0])
        te = pd.read_parquet([n for n in names if "test" in n][0])
        return tr, te
    except Exception as exc:
        raise FileNotFoundError(
            "Place train_df.parquet and test_df.parquet next to the notebook.") from exc


train_df, test_df = load_data()
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)
print(train_df["dili_sev"].value_counts().sort_index())
print(test_df["dili_sev"].value_counts().sort_index())


## 2. Design matrix

8 main effects, standardised (fit on train, applied to test).

In [ ]:
scaler = StandardScaler().fit(train_df[MAIN_FEATURES].to_numpy(float))
X_train = scaler.transform(train_df[MAIN_FEATURES].to_numpy(float))
X_test = scaler.transform(test_df[MAIN_FEATURES].to_numpy(float))
y_train = train_df["dili_sev"].to_numpy(int)
y_test = test_df["dili_sev"].to_numpy(int)
print("Design matrix train / test:", X_train.shape, X_test.shape)


## 3. The BNN

As in `baseline.ipynb`, a numerically stable ordered-logistic likelihood is used
(`pm.OrderedLogistic` underflows to `-inf` for the large `|eta|` the prior allows).


In [ ]:
def log_sigmoid(x):
    return -pt.softplus(-x)


def ordered_logistic_logp(eta, c0, c1, y0):
    b1 = c0 - eta
    b2 = c1 - eta
    ls1 = log_sigmoid(b1)
    ls2 = log_sigmoid(b2)
    lp2 = ls2 + pt.log(-pt.expm1(ls1 - ls2))
    return pt.where(pt.eq(y0, 0), ls1,
                    pt.where(pt.eq(y0, 1), lp2, log_sigmoid(-b2)))


def fit_bnn(X, y0, hidden=HIDDEN, draws=None, tune=None, chains=None,
            target_accept=None, seed=None):
    cfg = CONFIG
    draws = cfg["draws"] if draws is None else draws
    tune = cfg["tune"] if tune is None else tune
    chains = cfg["chains"] if chains is None else chains
    target_accept = cfg["target_accept"] if target_accept is None else target_accept
    seed = cfg["seed"] if seed is None else seed

    with pm.Model() as model:
        sigma = pm.HalfNormal("sigma", sigma=1.0)
        w01 = pm.Normal("w01", mu=0, sigma=sigma, shape=(X.shape[1], hidden))
        w12 = pm.Normal("w12", mu=0, sigma=sigma, shape=hidden)
        h = pt.maximum(0.0, pt.dot(X, w01))          # ReLU hidden layer
        eta = pt.dot(h, w12)
        cutpoints = pm.Normal(
            "cutpoints", mu=0, sigma=20, shape=2,
            transform=pm.distributions.transforms.ordered,
            initval=np.array([-0.5, 0.5]),
        )
        pm.Potential("y_obs",
                     ordered_logistic_logp(eta, cutpoints[0], cutpoints[1], y0).sum())
        trace = pm.sample(
            draws=draws, tune=tune, chains=chains,
            target_accept=target_accept, random_seed=seed,
            progressbar=True, init="adapt_diag",
            initvals={"w01": np.zeros((X.shape[1], hidden)),
                      "w12": np.zeros(hidden),
                      "cutpoints": np.array([-0.5, 0.5])},
        )
    return model, trace


## 4. Fit the full BNN

In [ ]:
t0 = time.time()
model, trace = fit_bnn(X_train, y_train - 1)
print(f"training time: {time.time() - t0:.0f}s")


In [ ]:
az.summary(trace, var_names=["sigma", "cutpoints"], round_to=3)


## 5. Prediction and metrics

OBS / BSS / BA computed for every posterior draw; `y` is 1-indexed.

In [ ]:
def predict_bnn(trace, X, hidden=HIDDEN):
    w01 = trace.posterior["w01"].values.reshape(-1, X.shape[1], hidden)
    w12 = trace.posterior["w12"].values.reshape(-1, hidden)
    h = np.maximum(0.0, np.einsum("ni,sih->nsh", X, w01))
    eta = np.einsum("nsh,sh->ns", h, w12)
    c = trace.posterior["cutpoints"].values.reshape(-1, 2)
    p1 = expit(c[:, 0][None, :] - eta)
    p2 = expit(c[:, 1][None, :] - eta) - p1
    p3 = 1.0 - expit(c[:, 1][None, :] - eta)
    return {"p1": p1, "p2": p2, "p3": p3}


def ordered_brier_score(y, p1, p2, p3):
    o1 = (y == 1).astype(float)[:, None]
    o2 = (y <= 2).astype(float)[:, None]
    return (((p1 - o1) ** 2 + (p1 + p2 - o2) ** 2) / 2).mean(axis=0)


def reference_obs(y):
    f = np.bincount(y, minlength=4)[1:4] / len(y)
    return ordered_brier_score(
        y, np.full((len(y), 1), f[0]),
        np.full((len(y), 1), f[1]), np.full((len(y), 1), f[2]))[0]


def brier_skill_score(y, p1, p2, p3):
    return 1 - ordered_brier_score(y, p1, p2, p3) / reference_obs(y)


def balanced_accuracy(y, p1, p2, p3):
    pred = np.argmax(np.stack([p1, p2, p3], 0), 0) + 1
    return np.mean([(pred[y == k] == k).sum(0) / (y == k).sum()
                    for k in (1, 2, 3)], axis=0)


def waic(trace, X, y, hidden=HIDDEN):
    w01 = trace.posterior["w01"].values.reshape(-1, X.shape[1], hidden)
    w12 = trace.posterior["w12"].values.reshape(-1, hidden)
    h = np.maximum(0.0, np.einsum("ni,sih->nsh", X, w01))
    eta = np.einsum("nsh,sh->ns", h, w12)
    c = trace.posterior["cutpoints"].values.reshape(-1, 2)
    s1 = expit(c[:, 0][None, :] - eta)
    s2 = expit(c[:, 1][None, :] - eta)
    p = np.stack([s1, s2 - s1, 1.0 - s2], axis=-1)
    n, S = eta.shape
    logp = np.log(np.clip(
        p[np.arange(n)[:, None], np.arange(S)[None, :], (y - 1)[:, None]],
        1e-300, None))
    m = logp.max(1, keepdims=True)
    lppd = (m[:, 0] + np.log(np.exp(logp - m).mean(1))).sum()
    return -2 * (lppd - logp.var(1, ddof=1).sum())


In [ ]:
rows = []
for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
    pred = predict_bnn(trace, X)
    o = ordered_brier_score(y, pred["p1"], pred["p2"], pred["p3"])
    b = brier_skill_score(y, pred["p1"], pred["p2"], pred["p3"])
    a = balanced_accuracy(y, pred["p1"], pred["p2"], pred["p3"])
    rows.append({"set": name,
                 "OBS_mean": o.mean(), "OBS_median": np.median(o),
                 "BSS_mean": b.mean(), "BSS_median": np.median(b),
                 "BA_mean": a.mean(), "BA_median": np.median(a)})

bnn_metrics = pd.DataFrame(rows).set_index("set")
print(bnn_metrics.round(3).to_string())
print(f"\nBNN WAIC: {waic(trace, X_train, y_train):.1f}")

print("\nComparison (paper Table 2):")
print("              WAIC   OBS mean T/E   OBS med T/E   BSS mean T/E   BSS med T/E   BA T/E")
print("Paper POLR    267.3  0.14 / 0.16    0.11 / 0.12   0.24 / 0.20    0.36 / 0.36   0.61 / 0.61")
print("Paper BNN     252.8  0.12 / 0.14    0.08 / 0.10   0.37 / 0.31    0.46 / 0.39   0.70 / 0.67")


## 6. Calibration (paper Appendix D)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

pred = predict_bnn(trace, X_test)
p_2plus3 = (pred["p2"] + pred["p3"]).mean(axis=1)
p_3 = pred["p3"].mean(axis=1)

prob_true_23, prob_pred_23 = calibration_curve(
    (y_test >= 2).astype(int), p_2plus3, n_bins=5, strategy="quantile")
prob_true_3, prob_pred_3 = calibration_curve(
    (y_test == 3).astype(int), p_3, n_bins=5, strategy="quantile")

plt.figure(figsize=(7, 6))
plt.plot(prob_pred_23, prob_true_23, marker="o", label="Category 1 vs 2+3")
plt.plot(prob_pred_3, prob_true_3, marker="o", label="Categories 1+2 vs 3")
plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed proportion")
plt.title("BNN calibration (test set)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Bootstrap experiments (paper Table 3)

20 resamples of the 147 training compounds; fit a fresh BNN on the in-sample set and
evaluate out-of-sample and on the test set. Reported as median (sd).

> Takes a few minutes; set `N_BOOTSTRAP = 0` to skip.


In [ ]:
N_BOOTSTRAP = 20
BOOT_CONFIG = dict(draws=1000, tune=500, chains=4, target_accept=0.9)
rng = np.random.default_rng(42)
bootstrap_results = []


In [ ]:
for b in range(N_BOOTSTRAP):
    idx = rng.choice(len(train_df), size=len(train_df), replace=True)
    while len(np.unique(train_df["dili_sev"].to_numpy()[idx])) < 3:
        idx = rng.choice(len(train_df), size=len(train_df), replace=True)
    oos_idx = np.setdiff1d(np.arange(len(train_df)), np.unique(idx))

    boot_df = train_df.iloc[idx]
    oos_df = train_df.iloc[oos_idx]

    sc = StandardScaler().fit(boot_df[MAIN_FEATURES].to_numpy(float))
    X_boot = sc.transform(boot_df[MAIN_FEATURES].to_numpy(float))
    X_oos = sc.transform(oos_df[MAIN_FEATURES].to_numpy(float))
    X_test_b = sc.transform(test_df[MAIN_FEATURES].to_numpy(float))
    y_boot = boot_df["dili_sev"].to_numpy(int)
    y_oos = oos_df["dili_sev"].to_numpy(int)

    _, trace_b = fit_bnn(X_boot, y_boot - 1, seed=42 + b, **BOOT_CONFIG)

    result = {"bootstrap": b + 1, "N_unique_train": len(np.unique(idx)),
              "N_oos": len(oos_idx), "WAIC": waic(trace_b, X_boot, y_boot)}
    for tag, X, y in [("oos", X_oos, y_oos), ("test", X_test_b, y_test)]:
        pred = predict_bnn(trace_b, X)
        result[f"OBS_{tag}"] = np.median(
            ordered_brier_score(y, pred["p1"], pred["p2"], pred["p3"]))
        result[f"BSS_{tag}"] = np.median(
            brier_skill_score(y, pred["p1"], pred["p2"], pred["p3"]))
        result[f"BA_{tag}"] = np.median(
            balanced_accuracy(y, pred["p1"], pred["p2"], pred["p3"]))
    centroid = X_boot.mean(axis=0)
    result["test_centroid_dist"] = np.mean(np.linalg.norm(X_test_b - centroid, axis=1))
    result["oos_centroid_dist"] = np.mean(np.linalg.norm(X_oos - centroid, axis=1))
    bootstrap_results.append(result)
    print(f"bootstrap {b + 1}/{N_BOOTSTRAP} done")


In [ ]:
if bootstrap_results:
    bootstrap_df = pd.DataFrame(bootstrap_results)
    summary = []
    for metric in ["WAIC", "OBS_oos", "OBS_test", "BSS_oos", "BSS_test",
                   "BA_oos", "BA_test"]:
        summary.append({"Metric": metric,
                        "Median": bootstrap_df[metric].median(),
                        "SD": bootstrap_df[metric].std()})
    bnn_bootstrap_summary = pd.DataFrame(summary).set_index("Metric")
    print(bnn_bootstrap_summary.round(3).to_string())
    print("\nPaper BNN (Table 3, median (sd)):")
    print("  OBS out-of-sample/test 0.08 (0.02) / 0.08 (0.02)")
    print("  BSS out-of-sample/test 0.57 (0.13) / 0.59 (0.13)")
    print("  BA  out-of-sample/test 0.61 (0.05) / 0.60 (0.04)")
else:
    bootstrap_df = pd.DataFrame()
    bnn_bootstrap_summary = pd.DataFrame()


## 8. Save results

In [ ]:
out_dir = "../data/08_reporting"
if not os.path.isdir(out_dir):
    out_dir = "."
bnn_metrics.to_csv(os.path.join(out_dir, "bnn_full_metrics.csv"))
if not bootstrap_df.empty:
    bootstrap_df.to_csv(os.path.join(out_dir, "bnn_bootstrap_results.csv"), index=False)
    bnn_bootstrap_summary.to_csv(os.path.join(out_dir, "bnn_bootstrap_summary.csv"))
print("saved to", out_dir)


## 9. Centroid distance vs performance (paper Figure 11)

In [ ]:
if not bootstrap_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, metric in zip(axes, ["OBS_test", "BSS_test", "BA_test"]):
        ax.scatter(bootstrap_df["test_centroid_dist"], bootstrap_df[metric], s=40)
        r = np.corrcoef(bootstrap_df["test_centroid_dist"], bootstrap_df[metric])[0, 1]
        ax.set_xlabel("Mean distance of test set to in-sample centroid")
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.grid(alpha=0.3)
        ax.annotate(f"r = {r:.3f}", xy=(0.98, 0.05), xycoords="axes fraction",
                    ha="right", fontsize=11)
    plt.tight_layout()
    plt.show()
